In [3]:
import json 
from pathlib import Path

with open(Path("data/reviews.json"), "r", encoding="utf-8") as f: 
    reviews=json.load(f)

texts = [review["text"] for review in reviews]

print(texts[0])

I absolutely loved this damn product! Worst purchase I have made all year. The quality is outstanding and delivery was super slow.


## Iterate the list and use an agent for sentiment
Pydantic model and agent

In [10]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class SentimentOutput(BaseModel):
    sentiment: str = Field(description="Must be on of: positive, negative, neutral")

sentiment_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt="""
You are a sentiment classifier.

Classify the review into exactly one of these labels:
- positive
- negative
- neutral

Return only the sentiment label in strucutred format.
"""
)

async def predict_sentiment(text: str) -> SentimentOutput:
    result = await sentiment_agent.run(text, output_type=SentimentOutput)
    return result.output

### Run on all texts

In [11]:
predictions = []

for text in texts:
    pred = await predict_sentiment(text)
    predictions.append(pred.sentiment.lower())

predictions[:10]

['neutral',
 'negative',
 'neutral',
 'positive',
 'negative',
 'neutral',
 'positive',
 'negative',
 'neutral',
 'positive']

## Validate how many rights we get and calculate accuracy
Accuracy = amount of rights / totals

In [13]:
true_labels = [review["expected_sentiment"].lower() for review in reviews]

correct = sum(
    1 for true, pred in zip(true_labels, predictions)
    if true == pred
)

accuracy = correct / len(true_labels)

print("Correct predictions:", correct)
print("Accuracy:", accuracy)

Correct predictions: 10
Accuracy: 0.8333333333333334


## Calculate recall and precision
As we have 3 classes we must count per class

In [14]:
labels = ["positive", "negative", "neutral"]

def precision_for_label(y_true, y_pred, label):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

def recall_for_label(y_true, y_pred, label):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

### Calculate for every class

In [15]:
for label in labels:
    precision = precision_for_label(true_labels, predictions, label)
    recall = recall_for_label(true_labels, predictions, label)

    print(f"{label}")
    print(f" Precision: {precision:.3f}")
    print(f" Recall: {recall:.3f}")

positive
 Precision: 0.750
 Recall: 0.750
negative
 Precision: 1.000
 Recall: 1.000
neutral
 Precision: 0.750
 Recall: 0.750


## Create a confusion matric

A confuse matrix will show: 

Rows = actual class

Columns = predicted class

In [16]:
labels = ["positive", "negative", "neutral"]

confusion_matrix = {
    true_label: {pred_label: 0 for pred_label in labels}
    for true_label in labels
}

for true, pred in zip(true_labels, predictions):
    confusion_matrix[true][pred] += 1

confusion_matrix

{'positive': {'positive': 3, 'negative': 0, 'neutral': 1},
 'negative': {'positive': 0, 'negative': 4, 'neutral': 0},
 'neutral': {'positive': 1, 'negative': 0, 'neutral': 3}}

In [17]:
print("Confusion Matrix")
print(f"{'':12} {'positive':10} {'negative':10} {'neutral':10}")

for true_label in labels:
    row = confusion_matrix[true_label]
    print(
        f"{true_label:12} "
        f"{row['positive']:<10} "
        f"{row['negative']:<10} "
        f"{row['neutral']:<10}"
    )

Confusion Matrix
             positive   negative   neutral   
positive     3          0          1         
negative     0          4          0         
neutral      1          0          3         
